# Push service checks — M3-13, M3-14, M3-12

Three cases against `server/callbacks/services/health_information_hiu_push_service.py`'s
`process_health_information_hiu_push()` — the HIU-side handler for the HIP's direct data push
(M3 Block 2, step 3). Same harness pattern as `set_a_idempotency.ipynb`: real service function,
isolated scratch storage, stubbed crypto/network.

**What's stubbed here specifically:** `decrypt_health_data` and `from_x509_public_key` (both real
ECDH/AES-GCM crypto — already covered by `tools/verify_fidelius.py`'s own round-trip test elsewhere;
these three cases are about consent-scope rejection and multi-page merge logic, not crypto correctness,
so the decrypt step is replaced with a simple stand-in that returns a fixed FHIR bundle string).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


---
## M3-13 — a pushed care context not covered by the consent is rejected, not silently stored

**Real-world scenario:** a HIP pushes health data for a `transactionId`. One entry's
`careContextReference` genuinely belongs to the consent that authorized this transfer; a second entry
carries a `careContextReference` that consent never granted — a bug on the HIP's side, stale/leftover
data, or a deliberately malicious push. Before this fix, `process_health_information_hiu_push()` decrypted
and stored whatever `careContextReference` a push claimed, with no check against what the consent
artefact (fetched and stored ourselves back in Block 1) actually authorized.

**Pass criteria:** the in-scope entry is decrypted and stored (`hi_status: OK`); the out-of-scope entry is
rejected (`hi_status: ERRORED`, not decrypted, not stored as real data).

In [ ]:
from unittest.mock import patch

import server.callbacks.services.health_information_hiu_push_service as push_service
from server.callbacks.repository.pending_health_information_request_repository import (
    save_pending_health_information_request, link_transaction_id,
)
from server.callbacks.repository.hiu_consent_repository import save_hiu_consent
from server.callbacks.repository.hiu_health_information_repository import get_hiu_health_information

harness.activate_scratch_storage("m3_13")

save_pending_health_information_request("req-A", {
    "consent_id": "consent-m3-13", "hip_id": "IN2810000123", "hiu_id": "HIU-1",
    "key_material": {"private_key": "our-priv", "nonce": "our-nonce"},
})
link_transaction_id("req-A", "txn-m3-13")
save_hiu_consent("consent-m3-13", {"consent_detail": {"careContexts": [{"careContextReference": "cc-in-scope"}]}})

notify_recorder = harness.CallRecorder(harness.FakeResponse(202))

def fake_decrypt(ciphertext, **kw):
    return f'{{"resourceType": "Bundle", "marker": "{ciphertext}"}}'

with patch.object(push_service, "decrypt_health_data", fake_decrypt), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE_HIP_PUBKEY"), \
     patch.object(push_service, "send_health_information_notify", notify_recorder):
    push_body = {
        "transactionId": "txn-m3-13", "pageNumber": 0, "pageCount": 1,
        "entries": [
            {"content": "ciphertext-in-scope", "checksum": push_service._compute_checksum("ciphertext-in-scope"), "careContextReference": "cc-in-scope"},
            {"content": "ciphertext-out-of-scope", "checksum": push_service._compute_checksum("ciphertext-out-of-scope"), "careContextReference": "cc-FABRICATED"},
        ],
        "keyMaterial": {"dhPublicKey": {"keyValue": "hip-pubkey-raw"}, "nonce": "hip-nonce"},
    }
    await push_service.process_health_information_hiu_push({"body": push_body})

stored = get_hiu_health_information("txn-m3-13")
harness.check("in-scope care context stored OK", stored["care_contexts"]["cc-in-scope"]["hi_status"] == "OK")
harness.check("out-of-scope (fabricated) care context rejected, not decrypted", stored["care_contexts"]["cc-FABRICATED"]["hi_status"] == "ERRORED")


---
## M3-14 — one malformed entry in a push must not sink the whole batch (CONFIRM ONLY)

The per-entry loop in `_decrypt_entries()` already wraps each entry's processing so a single bad entry
(checksum mismatch, decrypt failure) only marks that one `ERRORED` and continues to the next — no fix was
needed here, this cell just proves it's real rather than trusting the code on read alone.

**Pass criteria:** the good entry in the same batch is stored despite the corrupted one failing.

In [ ]:
harness.activate_scratch_storage("m3_14")

save_pending_health_information_request("req-B", {
    "consent_id": "consent-m3-14", "hip_id": "IN2810000123", "hiu_id": "HIU-1",
    "key_material": {"private_key": "our-priv", "nonce": "our-nonce"},
})
link_transaction_id("req-B", "txn-m3-14")
save_hiu_consent("consent-m3-14", {"consent_detail": {"careContexts": [
    {"careContextReference": "cc-good"}, {"careContextReference": "cc-corrupt"},
]}})

notify_recorder2 = harness.CallRecorder(harness.FakeResponse(202))
with patch.object(push_service, "decrypt_health_data", fake_decrypt), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE_HIP_PUBKEY"), \
     patch.object(push_service, "send_health_information_notify", notify_recorder2):
    push_body = {
        "transactionId": "txn-m3-14", "pageNumber": 0, "pageCount": 1,
        "entries": [
            {"content": "ciphertext-good", "checksum": push_service._compute_checksum("ciphertext-good"), "careContextReference": "cc-good"},
            {"content": "ciphertext-corrupt", "checksum": "0000-DELIBERATELY-WRONG", "careContextReference": "cc-corrupt"},
        ],
        "keyMaterial": {"dhPublicKey": {"keyValue": "hip-pubkey-raw"}, "nonce": "hip-nonce"},
    }
    await push_service.process_health_information_hiu_push({"body": push_body})

stored2 = get_hiu_health_information("txn-m3-14")
harness.check("good entry stored OK despite the other entry being corrupt", stored2["care_contexts"]["cc-good"]["hi_status"] == "OK")
harness.check("corrupt entry marked ERRORED (checksum mismatch), batch not silently dropped", stored2["care_contexts"]["cc-corrupt"]["hi_status"] == "ERRORED")


---
## M3-12 — a multi-page transfer merges across pages, notifies ABDM only once (CONFIRM ONLY)

**Real-world scenario:** a consent covers 2+ care contexts. Per the 2026-08-12 rework, each care context
is now pushed as its own page (own fresh encryption key — see that fix's own docstring for why one shared
key across a whole push is an AES-GCM nonce-reuse bug). This handler must accumulate care contexts across
pages for the same `transactionId` rather than overwriting, and must only tell ABDM the transfer
"completed" once, after the LAST page — not once per page.

**Pass criteria:** after both pages arrive, both care contexts are present in the merged record; the
notify call fires exactly once, only after the last page.

In [ ]:
harness.activate_scratch_storage("m3_12")

save_pending_health_information_request("req-C", {
    "consent_id": "consent-m3-12", "hip_id": "IN2810000123", "hiu_id": "HIU-1",
    "key_material": {"private_key": "our-priv", "nonce": "our-nonce"},
})
link_transaction_id("req-C", "txn-m3-12")
save_hiu_consent("consent-m3-12", {"consent_detail": {"careContexts": [
    {"careContextReference": "cc-page0"}, {"careContextReference": "cc-page1"},
]}})

notify_recorder3 = harness.CallRecorder(harness.FakeResponse(202))
with patch.object(push_service, "decrypt_health_data", fake_decrypt), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE_HIP_PUBKEY"), \
     patch.object(push_service, "send_health_information_notify", notify_recorder3):
    page0 = {"transactionId": "txn-m3-12", "pageNumber": 0, "pageCount": 2,
              "entries": [{"content": "c0", "checksum": push_service._compute_checksum("c0"), "careContextReference": "cc-page0"}],
              "keyMaterial": {"dhPublicKey": {"keyValue": "k"}, "nonce": "n"}}
    page1 = {"transactionId": "txn-m3-12", "pageNumber": 1, "pageCount": 2,
              "entries": [{"content": "c1", "checksum": push_service._compute_checksum("c1"), "careContextReference": "cc-page1"}],
              "keyMaterial": {"dhPublicKey": {"keyValue": "k"}, "nonce": "n"}}

    await push_service.process_health_information_hiu_push({"body": page0})
    harness.check("no notify yet after page 0 of 2", notify_recorder3.call_count == 0)

    await push_service.process_health_information_hiu_push({"body": page1})

stored3 = get_hiu_health_information("txn-m3-12")
harness.check("both pages' care contexts present in the merged record", set(stored3["care_contexts"].keys()) == {"cc-page0", "cc-page1"})
harness.check("notify fired exactly once total (only after the last page)", notify_recorder3.call_count == 1)
